In [ ]:
import torch
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")


CUDA available: True
GPU: Tesla T4
VRAM: 14.56 GB


In [1]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"

In [2]:
import torch
from unsloth import FastLanguageModel
max_seq_length = 2048
dtype = None
load_in_4bit = True

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [3]:
model,tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-3B-Instruct-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

==((====))==  Unsloth 2026.8.18: Fast Llama patching. Transformers: 5.13.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/Llama-3.2-3B-Instruct-bnb-4bit as a legacy tokenizer.


In [4]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth 2026.8.18 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [5]:
import json
file_path ="/content/diary_entries.jsonl"
raw_data=[]
with open(file_path,"r",encoding="utf-8") as f:
  for line in f:
    raw_data.append(json.loads(line))
print(raw_data[0])


{'input': 'I cleaned and reorganized my room and a wave of pure delight washed over me. I noticed it more than usual today.', 'output': 'Joyful'}


In [6]:
from datasets import Dataset

In [7]:
dataset = Dataset.from_list(raw_data)

In [9]:
dataset[100]

{'input': "I practiced the guitar for an hour and I couldn't shake a quiet sense of loss afterward. I keep coming back to that moment.",
 'output': 'Sad'}

In [18]:
def formatting_prompts_func(examples):
  convos=[]
  texts=[]
  system_prompt = "Classify the dominant emotional tone of this journal entry as one of: Joyful, Sad, Anxious, Calm."
  for input_text,output_text in zip(examples["input"],examples["output"]):
    conversation = [
        {"role":"system","content": system_prompt},
        {"role":"user","content": input_text},
        {"role":"assistant","content": output_text},
    ]
    text = tokenizer.apply_chat_template(
        conversation,
        tokenize=False,
        add_generation_prompt=False
    )
    texts.append(text)
  return {"text":texts}




In [19]:
dataset = dataset.map(formatting_prompts_func,batched = True)

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [21]:
dataset[0]['text']

'<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 26 July 2024\n\nClassify the dominant emotional tone of this journal entry as one of: Joyful, Sad, Anxious, Calm.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nI cleaned and reorganized my room and a wave of pure delight washed over me. I noticed it more than usual today.<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nJoyful<|eot_id|>'

In [24]:
from trl import SFTTrainer

In [26]:
from transformers import TrainingArguments

In [32]:
from unsloth import is_bfloat16_supported
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    #change dataset to arrow table type before giving to sfttrainer
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    datasewt_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 250,
        learning_rate = 1e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs"
    )

)

Unsloth: `push_to_hub_token` is not a valid SFTConfig argument for the installed TRL and will be IGNORED. Check the spelling, or your TRL version if this argument used to work.
Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/500 [00:00<?, ? examples/s]

In [33]:
trainer_status = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 500 | Num Epochs = 4 | Total steps = 250
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
1,3.978177
2,3.925324
3,3.863437
4,3.701919
5,3.343571
6,3.167985
7,2.940976
8,2.604447
9,2.429422
10,2.193088


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-250/tokenizer_config.json.


In [ ]:
#0.1-0.3

In [34]:
FastLanguageModel.for_inference(model)

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 3072, padding_idx=128004)
        (layers): ModuleList(
          (0-27): 28 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=3072, out_features=3072, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=3072, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=3072, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lor

In [42]:
test_data = [
    ("My roommate surprised me by leaving my favorite snack on my desk after a rough day, and I caught myself smiling for no particular reason.", "Joyful"),
    ("I opened the email expecting another rejection, but instead it invited me to the final interview. I felt a rush of excitement.", "Joyful"),
    ("My little brother remembered the silly nickname I gave him years ago, and the memory made the whole afternoon feel warm.", "Joyful"),
    ("I finally managed to play a difficult song without stopping, and I sat there grinning at myself afterward.", "Joyful"),
    ("A stranger returned the notebook I had accidentally left on the bus, and I felt unexpectedly grateful and cheerful.", "Joyful"),
    ("I woke up to sunlight filling the room and realized I had nothing urgent waiting for me. The morning felt wonderful.", "Joyful"),
    ("My friend sent me a ridiculous photo from our old trip, and I laughed so hard that my stomach hurt.", "Joyful"),
    ("I received a handwritten thank-you note from someone I had helped months ago, and it made me feel genuinely appreciated.", "Joyful"),
    ("The presentation I had practiced for days went better than expected, and I walked out feeling proud and energized.", "Joyful"),
    ("I discovered that the tiny plant I thought was dying had grown a new leaf, and somehow that made my entire day brighter.", "Joyful"),
    ("My favorite teacher remembered my name after a long break, and I felt strangely happy about such a small thing.", "Joyful"),
    ("We ended up dancing in the kitchen while waiting for dinner, and everyone was laughing too much to care about the food.", "Joyful"),
    ("I received a voice message from an old friend I had missed, and hearing their laugh instantly lifted my mood.", "Joyful"),

    ("I packed away some clothes that belonged to a phase of my life I still miss, and the room suddenly felt much emptier.", "Sad"),
    ("I saw a familiar place after several years, but it looked completely different. I felt a quiet ache for how it used to be.", "Sad"),
    ("My friend told me they were moving to another city, and I was happy for them but couldn't shake the sadness afterward.", "Sad"),
    ("I reread an old conversation tonight and realized how much things had changed between us. It left me feeling hollow.", "Sad"),
    ("I expected to enjoy the celebration, but I kept thinking about someone who couldn't be there anymore.", "Sad"),
    ("The song that played at dinner reminded me of a difficult year, and my mood dropped almost immediately.", "Sad"),
    ("I finished a movie everyone loved, but the ending left me with a strange heaviness that stayed for the rest of the night.", "Sad"),
    ("I found an unfinished drawing from when I was younger and felt a little grief for the person I used to be.", "Sad"),
    ("My plans for the weekend fell apart one by one, and by evening I just wanted to stay in bed.", "Sad"),
    ("I walked past the cafe where we used to meet every week, and seeing the empty table made me unexpectedly emotional.", "Sad"),
    ("I tried to celebrate my achievement, but the person I most wanted to tell wasn't around anymore.", "Sad"),
    ("The house was unusually quiet after everyone left, and I suddenly felt lonely despite enjoying the visit.", "Sad"),
    ("I looked through my childhood photos and felt grateful for those memories, but also deeply aware that those days were gone.", "Sad"),

    ("I have to present my idea tomorrow, and I keep imagining every possible way the audience could dislike it.", "Anxious"),
    ("My phone battery died while I was waiting for an important message, and I couldn't stop wondering whether something had gone wrong.", "Anxious"),
    ("I noticed a typo in the document right before submitting it and spent the next hour worrying that everyone would notice.", "Anxious"),
    ("The bus was late and I kept checking the time because I was convinced I would miss the appointment.", "Anxious"),
    ("I heard footsteps outside my room late at night and immediately started imagining all sorts of explanations.", "Anxious"),
    ("I submitted my application and now I keep refreshing the portal even though I know there is nothing new to see.", "Anxious"),
    ("My friend hasn't replied since yesterday, and my mind keeps inventing reasons why they might be upset with me.", "Anxious"),
    ("I was asked a question unexpectedly in class and my mind went blank while my heart started beating faster.", "Anxious"),
    ("The doctor said the results would take a few days, and waiting without knowing anything is making me uneasy.", "Anxious"),
    ("I have several things to finish before morning, and even while doing one task I keep worrying about all the others.", "Anxious"),
    ("I agreed to meet someone new, and as the time gets closer I keep wondering whether they will like me.", "Anxious"),
    ("I made a mistake during the group project and now I can't stop replaying the moment in my head.", "Anxious"),

    ("I sat beside the window with a cup of tea and listened to the rain without feeling the need to do anything else.", "Calm"),
    ("After finishing my chores, I turned off my phone and spent an hour reading. My mind felt unusually still.", "Calm"),
    ("The train ride was quiet, and watching the scenery pass by gave me a comfortable sense of peace.", "Calm"),
    ("I took a slow walk after dinner and noticed my breathing settle as the streets became quieter.", "Calm"),
    ("I organized my bookshelf while listening to soft music, and the simple routine made me feel settled.", "Calm"),
    ("There was no special event today, but everything moved at an easy pace and I felt content with that.", "Calm"),
    ("I spent the afternoon sketching without worrying about whether the drawing was good. It was relaxing to simply focus.", "Calm"),
    ("I sat on the balcony before sunrise and watched the sky change slowly. Nothing felt urgent.", "Calm"),
    ("After talking through a disagreement with my friend, I felt the tension leave and my thoughts became much clearer.", "Calm"),
    ("I cooked dinner slowly instead of rushing, and by the time I finished I felt grounded and comfortable.", "Calm"),
    ("I spent an hour cleaning my headphones and desk while listening to music, and the repetitive task was surprisingly soothing.", "Calm"),
    ("The library was nearly empty, and I settled into a corner where I could study without feeling distracted.", "Calm"),
]

In [51]:
y_true = []
y_pred = []
FastLanguageModel.for_inference(model)
model.generation_config.max_length = None
print("Running evaluation..")
print(f"{'Expected':<20} | {'Predicted (raw)':<20} | {'Match?'}")
print("-"*60)
for query,expected_lable in test_data:
  message = [
      {"role": "system", "content": "Classify the dominant emotional tone of this journal entry as one of: Joyful, Sad, Anxious, Calm."},
      {"role": "user", "content": query},
  ]
  inputs = tokenizer.apply_chat_template(message,
                                         tokenize=True,
                                         add_generation_prompt=True,
                                         return_tensors = "pt").to("cuda")
  output = model.generate(
      input_ids = inputs,
      max_new_tokens=10,
      use_cache=True,
      temperature=0.01)
  prediction = tokenizer.decode(
      output[0][inputs.shape[1]:],
      skip_special_tokens=True)
  print(f"{expected_lable:<20} | {repr(prediction):<20} | {expected_lable==prediction.strip()}")


Running evaluation..
Expected             | Predicted (raw)      | Match?
------------------------------------------------------------
Joyful               | 'Joyful'             | True
Joyful               | 'Joyful'             | True
Joyful               | 'Calm'               | False
Joyful               | 'Joyful'             | True
Joyful               | 'Joyful'             | True
Joyful               | 'Joyful'             | True
Joyful               | 'Joyful'             | True
Joyful               | 'Joyful'             | True
Joyful               | 'Joyful'             | True
Joyful               | 'Joyful'             | True
Joyful               | 'Joyful'             | True
Joyful               | 'Joyful'             | True
Joyful               | 'Joyful'             | True
Sad                  | 'Sad'                | True
Sad                  | 'Sad'                | True
Sad                  | 'Sad'                | True
Sad                  | 'Sad'                | Tr

In [59]:
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')

In [60]:
from huggingface_hub import login

login(HF_TOKEN)

In [62]:
model.push_to_hub_merged(
    "swasthika28/llama-3.2-3b-mood-classifier",
    tokenizer,
    save_method="merged_16bit"
)

No files have been modified since last commit. Skipping to prevent empty commit.
Unsloth: Restored added_tokens_decoder metadata in /tmp/tmpp3xrf2nj/tokenizer_config.json.
No files have been modified since last commit. Skipping to prevent empty commit.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

No files have been modified since last commit. Skipping to prevent empty commit.


...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...0002-of-00002.safetensors:   1%|1         | 15.9MB / 1.46GB            





Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [02:39<00:00, 79.59s/it]


Unsloth: Merge process complete. Saved to `/tmp/tmpp3xrf2nj`
